<a href="https://colab.research.google.com/github/edmundzen/dengue-early-warning/blob/main/member4_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai

**MEMBER 4 : Gemini AI & Integration**

Integrate the Gemini API.
Generate AI recommendations/explanations based on the risk scores.
Help connect all the modules and test the complete workflow.
## Tasks Completed
- Connected to Google BigQuery.
- Read the `inspection_priority` table generated by Member 2.
- Integrated the Gemini API.
- Generated AI recommendations for high-risk dengue areas.
- Saved the recommendations as a CSV (`inspection_with_ai.csv`).
- Verified the output for dashboard integration.

## Workflow
BigQuery (inspection_priority)
⬇
Gemini API
⬇
AI Recommendations
⬇
CSV Output
⬇
Looker Studio Dashboard

## Step 1: Import Required Libraries

In [1]:
from google import genai
from google.genai import types
import pandas as pd
from google.cloud import bigquery
from google.colab import auth
from google.colab import auth, userdata

## Step 2: Authenticate with Google Cloud

In [2]:
auth.authenticate_user()

import subprocess
acct = subprocess.run(["gcloud","config","get-value","account"],
                      capture_output=True, text=True).stdout.strip()
print("Signed in as:", acct)

Signed in as: shraddhapal1505@gmail.com


## Step 3: Connect to BigQuery

In [3]:
PROJECT_ID = "dengue-early-warning"
client = bigquery.Client(project=PROJECT_ID)
print("Connected!")

Connected!


## Step 4: Load the `inspection_priority` Table

In [4]:
query = """
SELECT * FROM `dengue-early-warning.dengue_ew.inspection_priority`
WHERE risk_score > 0 ORDER BY risk_score DESC LIMIT 20
"""
inspection_df = client.query(query).to_dataframe()
print("Loaded", len(inspection_df), "cells")

Loaded 20 cells


In [5]:
print(inspection_df["risk_score"].tolist())

[24.73, 24.61, 22.84, 22.08, 21.92, 21.88, 21.85, 21.79, 21.77, 21.74, 21.54, 21.28, 21.27, 21.22, 21.14, 20.76, 20.63, 20.59, 20.57, 20.47]


## Step 5: Configure Gemini API

In [23]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_KEY")   # Use the secret NAME
client = genai.Client(api_key=api_key)

print("Gemini ready!")

Gemini ready!


In [24]:
from google import genai

def generate_recommendation(row):
    prompt = f"""
You are a public health officer.

Based on the following dengue inspection data, provide 2-3 short recommendations.

Cell ID: {row['cell_id']}
Risk Score: {row['risk_score']}

Keep the recommendations practical and concise.
"""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return response.text

    except Exception as e:
        return f"Error: {e}"

In [26]:
from google import genai

client = genai.Client(api_key=api_key)

In [29]:
import google.genai
print("google-genai imported successfully")

google-genai imported successfully


In [34]:
from google import genai

client = genai.Client(api_key=api_key)

In [58]:
from google import genai

client = genai.Client(api_key="GEMINI_KEY")

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Hello"
)

print(response.text)

Hello there! How can I help you today?


In [56]:
from google.colab import userdata

api_key = userdata.get("GEMINI_KEY")
print(api_key[:8])
print(len(api_key))

AQ.Ab8RN
53


In [42]:
inspection_df = inspection_df.head(3).copy()

In [43]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Hello"
)

print(response.text)

Hi there! How can I help you today?


In [44]:
response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Say Hello"
)

print(response.text)

Hello! How can I help you today?


In [46]:
response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Hello"
)

print(response.text)

Hi there! How can I help you today?


## Step 6: Generate AI Recommendations

In [47]:
def generate_recommendation(row):
    prompt = f"""
You are a public health officer.

Based on the following data:
Cell ID: {row['cell_id']}
Risk Score: {row['risk_score']}

Give 2 short recommendations.
"""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"Error: {e}"

In [49]:
inspection_df = inspection_df.head(3).copy()
inspection_df["AI_Recommendation"] = inspection_df.apply(
    generate_recommendation,
    axis=1
)

inspection_df[["cell_id", "risk_score", "AI_Recommendation"]]

,cell_id,risk_score,AI_Recommendation
0,1.33875_103.86,24.73,"As a public health officer, based on the provi..."
1,1.33875_103.86,24.61,"As a public health officer, based on the provi..."
2,1.33875_103.85775,22.84,Here are two short recommendations based on th...


In [51]:
from google.cloud import bigquery

bq_client = bigquery.Client(project="dengue-early-warning")

## Step 7: Export Recommendations to CSV

In [52]:
inspection_df.to_csv("inspection_with_ai.csv", index=False)
print("CSV saved")

from google.cloud import bigquery

bq_client = bigquery.Client(project="dengue-early-warning")

job = bq_client.load_table_from_dataframe(
    inspection_df,
    "dengue-early-warning.dengue_ew.inspection_with_ai",
    job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"
    ),
)

job.result()

print("Uploaded", job.output_rows, "rows")

CSV saved
Uploaded 3 rows


# Conclusion

The Gemini AI module successfully converts dengue risk scores into actionable recommendations for health officials. These recommendations can be integrated into the dashboard to support faster and better decision-making during dengue outbreaks.